# 3 — From Graph to Spreadsheet and Back — Solutions

This file contains reference solutions for all 9 exercises in notebook 3.

## Setup (run before any exercise)

In [ ]:
import bw2data as bd
import bw2io as bi
from pathlib import Path
import pandas as pd
import openpyxl
from bw2io.strategies import link_iterable_by_fields
from copy import deepcopy

bd.projects.set_current("BAFU 2025 v2")

In [ ]:
# Export the garden to Excel (needed for all later exercises)
filepath = Path.cwd() / "swiss_garden_foreground.xlsx"
bi.export.write_lci_excel("Swiss garden foreground", filepath=str(filepath))
print(f"Exported to {filepath}")

In [ ]:
# Create the broken version used by the ETL exercises
wb = openpyxl.load_workbook("swiss_garden_foreground.xlsx")
ws = wb.active

name_col = None
unit_col = None
unit_changed = False

for row in ws.iter_rows():
    for cell in row:
        if cell.value == "name":
            name_col = cell.column
        if cell.value == "unit":
            unit_col = cell.column

for row in ws.iter_rows(min_row=2):
    if name_col and row[name_col - 1].value == "Garden construction":
        row[name_col - 1].value = "Garden construction process"
    if unit_col and not unit_changed and row[unit_col - 1].value == "unit":
        row[unit_col - 1].value = "pieces"
        unit_changed = True

wb.save("swiss_garden_broken.xlsx")
print("Saved swiss_garden_broken.xlsx")

---

## Exercise 1 — Inspect the Excel file

**Question**: What columns are present? What does each row represent? How are processes separated from exchanges?

**Solution**:

In [ ]:
# The ExcelImporter format uses a specific layout:
# - Rows starting with 'Activity' mark the beginning of a new process dataset
# - The header row immediately follows and names the columns for the exchanges below
# - Subsequent rows are exchanges belonging to that process
df = pd.read_excel("swiss_garden_foreground.xlsx", sheet_name=0, header=None)
df.head(40)

In [ ]:
# Key observations:
# 1. The file has blocks separated by rows containing 'Activity' in the first column.
#    Each block is one process (dataset).
# 2. Within each block, a sub-header row names the exchange fields:
#    name, amount, unit, type, location, categories, etc.
# 3. Each subsequent row is one exchange (technosphere input, biosphere emission, or production).
# 4. Background inputs (e.g. gravel, excavation) are identified by their name, location, unit,
#    and which database they belong to — but NOT by any pointer or code.
#    This is exactly the BOM paradigm: attributes, not pointers.

print("The file uses blocks separated by 'Activity' rows.")
print("Each block = one process. Each subsequent row = one exchange.")
print("Background inputs are identified by name + location + unit — no direct pointer.")

---

## Exercise 2 — Statistics after extraction

**Question**: How many datasets were extracted? How many total exchanges?

In [ ]:
imp = bi.ExcelImporter(Path.cwd() / "swiss_garden_broken.xlsx")
imp.statistics()

In [ ]:
# Manual check
n_datasets = len(imp.data)
n_exchanges = sum(len(ds['exchanges']) for ds in imp.data)
print(f"Datasets (processes): {n_datasets}")
print(f"Total exchanges:      {n_exchanges}")
print()
print("The Swiss garden foreground has 5 processes (Garden system, Garden construction,")
print("Annual garden maintenance, Annual green waste treatment, Garden end-of-life)")
print("plus product nodes = 10 nodes total. Exchanges include production + technosphere inputs.")

---

## Exercise 3 — Unlinked exchanges after default strategies

**Question**: What does 'unlinked' mean? What are the unlinked exchanges?

In [ ]:
imp.apply_strategies()
imp.statistics()

In [ ]:
# "Unlinked" means the exchange dict does not yet have an 'input' key.
# A linked exchange has: {'input': ('database_name', 'node_code'), 'amount': ..., ...}
# An unlinked exchange has: {'name': '...', 'location': '...', 'amount': ..., ...}
#                           (no 'input' field)

print("Unlinked exchanges:")
for edge in imp.unlinked:
    print(" ", edge)

In [ ]:
# Analysis:
# - The 'Garden construction process' exchange is unlinked because we renamed
#   the target node, so name-based lookup fails.
# - The exchange with unit 'pieces' is unlinked because the unit does not match
#   the corresponding node's unit.
# - All background (BAFU) exchanges are also unlinked because the default strategies
#   only attempt internal linking within the imported database, not against BAFU.

print("Causes of unlinking:")
print("1. Name mismatch: 'Garden construction process' vs. 'Garden construction'")
print("2. Unit mismatch: 'pieces' vs. 'unit'")
print("3. Background exchanges: default strategies don't know to look in BAFU-2025-v2")

---

## Exercise 4 — Fix the unit mismatch

**Task**: Write a transformation to change `"pieces"` → `"unit"` on affected exchanges.

In [ ]:
# First fix the name (from the notebook)
def fix_construction_name(data):
    for ds in data:
        for exc in ds['exchanges']:
            if exc.get('name') == 'Garden construction process':
                exc['name'] = 'Garden construction'
    return data

imp.apply_strategy(fix_construction_name)

# Now fix the unit
def fix_unit_mismatch(data):
    """Change any exchange unit of 'pieces' back to 'unit'."""
    for ds in data:
        for exc in ds['exchanges']:
            if exc.get('unit') == 'pieces':
                exc['unit'] = 'unit'
    return data

imp.apply_strategy(fix_unit_mismatch)
imp.statistics()

In [ ]:
# Note: you can make the fix more targeted (only change if unlinked and unit=='pieces')
# but the broad version is fine for this dataset because we know the context.
# A more defensive version:

def fix_unit_mismatch_targeted(data):
    """Change 'pieces' → 'unit' only on unlinked exchanges."""
    for ds in data:
        for exc in ds['exchanges']:
            if 'input' not in exc and exc.get('unit') == 'pieces':
                exc['unit'] = 'unit'
    return data

# (Would work identically here; included for illustration)

---

## Exercise 5 — Why match foreground first? What happens with BAFU?

**Question**: Why `"Swiss garden foreground"` first? What happens when you also try BAFU?

In [ ]:
# Link foreground-to-foreground first
imp.match_database("Swiss garden foreground", fields=['name', 'location'])
imp.statistics()

In [ ]:
# Now link against BAFU
imp.match_database("BAFU-2025-v2", fields=['name', 'location'])
imp.statistics()

In [ ]:
# Explanation:
#
# We match the foreground first because:
#   1. Foreground-to-foreground links are unambiguous (we control both sides).
#   2. If we matched against BAFU first, a foreground process with the same name
#      as a BAFU process could be wrongly linked to BAFU.
#   3. Matching order matters: once an exchange has an 'input', it is skipped
#      by subsequent match_database calls (unless relink=True).
#
# After matching BAFU:
#   - Many background inputs will now be linked (excavation, transport, gravel, etc.).
#   - If name+location is insufficient (e.g., two BAFU processes share both), some
#     exchanges may remain unlinked — which is safer than a wrong link.

print("Match foreground first: avoids foreground processes being linked to wrong BAFU entry.")
print("match_database skips already-linked exchanges by default.")

---

## Exercise 6 — Field sufficiency for BAFU

**Question**: Is `name + location` always enough to uniquely identify a BAFU node? What could go wrong?

In [ ]:
# Check if any BAFU processes share the same (name, location)
bg = bd.Database("BAFU-2025-v2")

from collections import Counter
name_loc_pairs = Counter(
    (node.get('name', ''), node.get('location', ''))
    for node in bg
)
duplicates = {k: v for k, v in name_loc_pairs.items() if v > 1}
print(f"(name, location) pairs with more than one node: {len(duplicates)}")
for pair, count in list(duplicates.items())[:10]:
    print(f"  {pair}: {count} nodes")

In [ ]:
# Discussion:
#
# In most LCI databases, (name, location) is NOT guaranteed to be unique.
# A multi-output process may have multiple nodes with the same name and location
# but different reference products.
#
# Adding 'unit' helps: name + location + unit is more specific.
# Adding 'reference product' is even better in databases that use the product-process split.
#
# Using too many fields:
# - If an exchange lacks a field (e.g., no 'reference product' in the spreadsheet),
#   the key built from those fields will be None/missing for that field,
#   and the match will fail even if the name and location are correct.
# - So: use the minimum set of fields that achieves uniqueness.
#
# For BAFU: name + location + unit is usually sufficient and safe.

print("Safe strategy for BAFU: fields=['name', 'location', 'unit']")

In [ ]:
# Check unlinked after the two match passes
print("Still unlinked:")
for edge in imp.unlinked:
    print(" ", edge)

---

## Exercise 7 — `link_iterable_by_fields` with multiple field combinations

In [ ]:
unlinked_ex7 = [
    {
        "name": "Electricity Production, Natural Gas",
        "location": "DE",
        "database": "student_db",
        "code": "elec_ng_de",
        "exchanges": [
            {"name": "Natural Gas", "location": "RU", "unit": "MJ",  "amount": 1.2, "type": "technosphere"},
            {"name": "Natural Gas", "location": "NO", "unit": "MJ",  "amount": 0.3, "type": "technosphere"},
            {"name": "Methane",     "categories": ("air",), "unit": "kg", "amount": 0.04, "type": "biosphere"},
            {"name": "Carbon Dioxide", "categories": ("air", "unspecified"), "unit": "kg", "amount": 0.5, "type": "biosphere"},
        ]
    }
]

target_ex7 = [
    {"name": "Natural Gas", "location": "RU", "database": "ecoinvent", "code": "ng_ru"},
    {"name": "Natural Gas", "location": "NO", "database": "ecoinvent", "code": "ng_no"},
    {"name": "Methane",     "categories": ("air",), "database": "biosphere3", "code": "ch4_air"},
    {"name": "Carbon Dioxide", "categories": ("air", "unspecified"), "database": "biosphere3", "code": "co2_unspec"},
]

In [ ]:
# Pass 1: link technosphere exchanges by name + location
result_ex7 = link_iterable_by_fields(
    deepcopy(unlinked_ex7),
    other=target_ex7,
    fields=["name", "location"],
    edge_kinds=["technosphere"]
)

# Pass 2: link biosphere exchanges by name + categories
result_ex7 = link_iterable_by_fields(
    result_ex7,
    other=target_ex7,
    fields=["name", "categories"],
    edge_kinds=["biosphere"]
)

# Verify
for exc in result_ex7[0]['exchanges']:
    status = exc.get('input', 'UNLINKED')
    print(f"  {exc['name']:30s}  →  {status}")

In [ ]:
# Key insight: you cannot link ALL exchanges in one call if they need different fields.
# Technosphere: name + location (no categories field)
# Biosphere: name + categories (no location field)
# Attempting to use fields=['name', 'location', 'categories'] on biosphere exchanges
# would fail because biosphere entries in target have no 'location' field — the key
# would not match.
print("Two separate calls needed: technosphere uses location, biosphere uses categories.")

---

## Exercise 8 — Implications of dropping unlinked biosphere edges

**Scenario**: 3 unlinked biosphere flows (CO₂, CH₄, NOₓ). You call `drop_unlinked(i_am_reckless=True)`.

In [ ]:
# 1. Climate score:
#    CO₂ and CH₄ are both climate-relevant. Dropping them removes their contribution
#    from the inventory. The LCIA score will be LOWER than the true value —
#    it is artificially underestimated. The error can be significant.

# 2. Air quality score:
#    NOₓ is a key air pollutant. Dropping it gives an incomplete result.
#    Similarly underestimated.

# 3. Unlinked technosphere exchanges:
#    Dropping an unlinked technosphere exchange means the entire sub-system
#    upstream of that exchange is excluded from the calculation.
#    This affects BOTH the inventory (all upstream emissions are lost)
#    AND the LCIA score (potentially a large underestimate if the missing
#    process is a major contributor).
#    This is generally more damaging than dropping a biosphere flow.

# 4. When dropping might be acceptable:
#    - The unlinked flow has a very small amount and no characterisation factor
#      exists in the LCIA method anyway (it would contribute zero regardless)
#    - You are doing a screening assessment where precision is not critical
#    - The flow is from an auxiliary/infrastructure input that is genuinely
#      negligible (< 0.1% of total score by prior knowledge)
#    - ALWAYS document what you dropped and why in your LCA report

print("Dropping unlinked biosphere edges silently removes emissions from the inventory.")
print("The LCIA score will be underestimated. Always prefer creating a new biosphere")
print("database so the flows are at least recorded, even if not characterised.")

---

## Exercise 9 — Capstone: export and re-import the steel bicycle

**Task**: Export the steel bicycle from notebook 1 to Excel, import it back, fix unlinked exchanges.

In [ ]:
# Step 1 — Export
# (Assumes a database called 'Steel bicycle' exists in the current project.
#  Adjust the name to match your notebook 1 database.)

bicycle_db_name = "Steel bicycle"  # <-- change if needed

if bicycle_db_name in bd.databases:
    bicycle_filepath = Path.cwd() / "steel_bicycle.xlsx"
    bi.export.write_lci_excel(bicycle_db_name, filepath=str(bicycle_filepath))
    print(f"Exported {bicycle_db_name} to {bicycle_filepath}")
else:
    print(f"Database '{bicycle_db_name}' not found. Available: {list(bd.databases)}")

In [ ]:
# Step 2 — Extract
bike_imp = bi.ExcelImporter(Path.cwd() / "steel_bicycle.xlsx")
print("Raw data sample:")
bike_imp.data[0]

In [ ]:
# Step 3 — Transform (default strategies)
bike_imp.apply_strategies()

# Step 4 — Statistics
bike_imp.statistics()

In [ ]:
# Step 5 — Inspect unlinked
print("Unlinked exchanges:")
for edge in bike_imp.unlinked:
    print(" ", edge)

In [ ]:
# Step 6 — Match against relevant databases
# The bicycle foreground references itself internally, and may reference
# the same background database as the garden (BAFU-2025-v2 or ecoinvent).

# Internal foreground links
bike_imp.match_database(bicycle_db_name, fields=['name', 'location'])
bike_imp.statistics()

# Background links — adjust database name as appropriate
# bike_imp.match_database("BAFU-2025-v2", fields=['name', 'location'])
# bike_imp.match_database("ecoinvent-3.10.1-cutoff", fields=['name', 'reference product', 'location'])
# bike_imp.statistics()

In [ ]:
# Step 7 — Write
# If there are still unlinked biosphere edges, create a new database for them
if not bike_imp.all_linked:
    unlinked_list = list(bike_imp.unlinked)
    biosphere_unlinked = [e for e in unlinked_list if e.get('type') == 'biosphere']
    technosphere_unlinked = [e for e in unlinked_list if e.get('type') == 'technosphere']

    print(f"Unlinked biosphere: {len(biosphere_unlinked)}")
    print(f"Unlinked technosphere: {len(technosphere_unlinked)}")

    if biosphere_unlinked and not technosphere_unlinked:
        bike_imp.create_new_biosphere("bicycle-new-biosphere")
    elif technosphere_unlinked:
        print("WARNING: unlinked technosphere edges remain. Fix these before writing.")

if bike_imp.all_linked:
    bike_imp.write_database()
    print(f"Database '{bicycle_db_name}' written.")
    print(bd.databases)

In [ ]:
# Discussion of typical findings:
#
# A simple bicycle system built with explicit Python edges (notebook 1) should
# round-trip cleanly if the background database names and codes are consistent.
# Unlinked exchanges after export+import are most often caused by:
#
# a) The exported file records exchanges by name+location, but the original
#    edge was created with a Python pointer — the name stored in the Excel file
#    must exactly match the name field of the target node.
#
# b) Background nodes (BAFU, ecoinvent) are referenced by name in the Excel file
#    but must be re-linked against the live database. If the name stored in the
#    background database has trailing spaces or different capitalisation, linking fails.
#
# c) The database name column in the Excel may not be set correctly, causing
#    the importer to look in the wrong database.

print("Round-trip is clean when names and locations are stored consistently.")
print("Check the 'database' column in the Excel for background exchanges.")

---

## Bonus: `link_iterable_by_fields` with node-type filtering (from Section 5)

In [ ]:
# Solution to the node-type filtering exercise
node_type_data = [
    {
        "name": "Steel Production Process",
        "type": "process",
        "database": "my_db",
        "code": "steel_proc_001",
        "exchanges": [
            {"name": "Steel Product",        "amount": 1.0, "type": "technosphere"},
            {"name": "Iron Ore Product",     "amount": 1.5, "type": "technosphere"},
            {"name": "Another Steel Process","amount": 0.2, "type": "technosphere"},
        ]
    },
    {"name": "Steel Product",          "type": "product",  "database": "my_db", "code": "steel_prod_001",   "exchanges": []},
    {"name": "Iron Ore Product",       "type": "product",  "database": "my_db", "code": "iron_ore_prod_001","exchanges": []},
    {"name": "Another Steel Process",  "type": "process",  "database": "my_db", "code": "steel_proc_002",   "exchanges": []},
    {"name": "Carbon Dioxide",         "type": "elementary","categories": ("air",), "database": "biosphere3", "code": "co2_air", "exchanges": []},
]

result = link_iterable_by_fields(
    deepcopy(node_type_data),
    internal=True,
    fields=["name"],
    this_node_kinds=["process"],   # only process nodes participate as sources
    other_node_kinds=["product"]   # only product nodes can be targets
)

# Verify: Steel Product and Iron Ore Product should be linked; Another Steel Process should NOT be
for exc in result[0]['exchanges']:
    print(f"  {exc['name']:30s}  →  {exc.get('input', 'UNLINKED')}")

In [ ]:
# Expected output:
#   Steel Product                  →  ('my_db', 'steel_prod_001')
#   Iron Ore Product               →  ('my_db', 'iron_ore_prod_001')
#   Another Steel Process          →  UNLINKED   ← correct! blocked by other_node_kinds=['product']

print("'Another Steel Process' correctly remains unlinked — only product nodes are valid targets.")